In [2]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types

In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/10 12:48:29 WARN Utils: Your hostname, codespaces-37f61e, resolves to a loopback address: 127.0.0.1; using 10.0.3.50 instead (on interface eth0)
26/05/10 12:48:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/10 12:48:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark.version

'4.1.1'

In [4]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-05-10 09:36:48--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.239.115.146, 18.239.115.213, 18.239.115.4, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.239.115.146|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet.1’

yellow_tripdata_202 100%[===================>]  67.84M   254MB/s    in 0.3s    

2026-05-10 09:36:49 (254 MB/s) - ‘yellow_tripdata_2025-11.parquet.1’ saved [71134255/71134255]



In [4]:
df = spark.read.parquet('yellow_tripdata_2025-11.parquet')
df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [6]:
df.repartition(4)
df.write.parquet('yellow_tripdata_2025-11_repartitioned')

AnalysisException: [PATH_ALREADY_EXISTS] Path file:/workspaces/DataEngineering-DataTalksClub/06-batch/code/yellow_tripdata_2025-11_repartitioned already exists. Set mode as "overwrite" to overwrite the existing path. SQLSTATE: 42K04

<h4>Question 1:</h4>
<h5>What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? Select the answer which most closely matches. ?</h5>

-l: long format
<p>-h: human-readable</p>
!ls -lh path/to/your/dataframe.parquet/

In [6]:
!ls -lh /workspaces/DataEngineering-DataTalksClub/06-batch/code/yellow_tripdata_2025-11_repartitioned/part-00000-0f5f5efb-fc29-4716-9f48-0e8cf2b2dd54-c000.snappy.parquet

-rw-r--r-- 1 vscode vscode 21M May 10 06:45 /workspaces/DataEngineering-DataTalksClub/06-batch/code/yellow_tripdata_2025-11_repartitioned/part-00000-0f5f5efb-fc29-4716-9f48-0e8cf2b2dd54-c000.snappy.parquet


!du -sh path/to/your/dataframe.parquet/

! - bash terminal
du - disk usage (gets file size of ..)
-sh summary human-readable

In [12]:
!du -h /workspaces/DataEngineering-DataTalksClub/06-batch/code/yellow_tripdata_2025-11_repartitioned/part-00000-0f5f5efb-fc29-4716-9f48-0e8cf2b2dd54-c000.snappy.parquet

21M	/workspaces/DataEngineering-DataTalksClub/06-batch/code/yellow_tripdata_2025-11_repartitioned/part-00000-0f5f5efb-fc29-4716-9f48-0e8cf2b2dd54-c000.snappy.parquet


<strong>The average size of each parquet from our partitions is 25MB , actual = 21MB</strong>

<h4>Question 2:</h4>
<h5>How many taxi trips were there on the 15th of November?</h5>

<p>Consider only trips that started on the 15th of November.?</p>

In [19]:
nov_records = df.select('*').filter('tpep_pickup_datetime > "2025-11-15"').count()
print(f'No of records in starting mid-November: {nov_records}')

No of records in starting mid-November: 2133304


In [13]:
num_records = df.count()
print(f'No of records: {num_records}')

No of records: 4181444


<strong>The no of records starting mid november are 2,133,304</strong>

<h4>Question 3:</h4>
<h5>What is the length of the longest trip in the dataset in hours?</h5>

In [5]:
df_duration = df.select('tpep_pickup_datetime', 'tpep_dropoff_datetime', 'trip_distance')
df_duration.show(5)

+--------------------+---------------------+-------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|
+--------------------+---------------------+-------------+
| 2025-11-01 00:13:25|  2025-11-01 00:13:25|         1.68|
| 2025-11-01 00:49:07|  2025-11-01 01:01:22|         2.28|
| 2025-11-01 00:07:19|  2025-11-01 00:20:41|          2.7|
| 2025-11-01 00:00:00|  2025-11-01 01:01:03|        12.87|
| 2025-11-01 00:18:50|  2025-11-01 00:49:32|          8.4|
+--------------------+---------------------+-------------+
only showing top 5 rows


In [6]:
df_duration.registerTempTable('duration')

/workspaces/DataEngineering-DataTalksClub/venv/lib/python3.10/site-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


| Function | Description |
| :--- | :--- |
| `current_date()` / `current_timestamp()` | Gets the system date or time. |
| `date_add(col, n)` | Adds $n$ days to a date. |
| `datediff(end, start)` | Returns difference in days as an integer. |
| `date_trunc(fmt, col)` | Returns timestamp truncated to the unit specified by `fmt`. |
| `year()`, `month()`, `dayofweek()` | Extracts that specific component as an integer. |
| `last_day(col)` | Returns the last day of the month for a given date. |

using unix_timestamp() - returns timestamp in seconds

In [22]:
df_duration = spark.sql("""
    select tpep_pickup_datetime, tpep_dropoff_datetime, ((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600) as trip_duration_h, trip_distance
    from duration
    sort by trip_duration_h desc
""")

In [27]:
longest_trip = df_duration.first()
print(f'Longest trip duration in hours: {longest_trip.trip_duration_h:.2f}')

Longest trip duration in hours: 76.21


In [28]:
df_duration.show(5)

+--------------------+---------------------+------------------+-------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|   trip_duration_h|trip_distance|
+--------------------+---------------------+------------------+-------------+
| 2025-11-03 10:42:55|  2025-11-06 14:55:45| 76.21388888888889|          0.0|
| 2025-11-07 11:23:22|  2025-11-10 08:40:41| 69.28861111111111|         6.54|
| 2025-11-05 14:49:55|  2025-11-07 09:33:09|42.720555555555556|          0.0|
| 2025-11-06 18:34:34|  2025-11-08 12:11:26|41.614444444444445|          5.6|
| 2025-11-05 09:51:42|  2025-11-06 23:46:22| 37.91111111111111|        16.47|
+--------------------+---------------------+------------------+-------------+
only showing top 5 rows


<h4>Question 5:</h4>
<h5>Spark's User Interface which shows the application's dashboard runs on which local port?</h5>

sparks dashboard is run on port 4040

<h4>Question 6: Least frequent pickup location zone</h4>
<h5>Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?</h5>

In [29]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-05-10 13:13:08--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.239.115.213, 18.239.115.146, 18.239.115.4, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.239.115.213|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv.1’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-05-10 13:13:08 (168 MB/s) - ‘taxi_zone_lookup.csv.1’ saved [12331/12331]



In [30]:
df_taxi_zone = spark.read.csv('taxi_zone_lookup.csv', header=True, inferSchema=True)
df_taxi_zone.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


so ill have to join yellow data + the lookup

In [31]:
df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [32]:
df_join = df.join(df_taxi_zone, on=df.PULocationID == df_taxi_zone.LocationID, how='left')
df_join.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----------+---------+-------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|LocationID|  Borough|               Zone|service_zone|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----------+---------+--

In [38]:
frequent_pickup_zone = df_join.select('zone', 'LocationID').groupBy('zone').count().orderBy('count', ascending=False)
frequent_pickup_zone.show(5)

+--------------------+------+
|                zone| count|
+--------------------+------+
|Upper East Side S...|194593|
|      Midtown Center|182907|
|Upper East Side N...|169621|
|         JFK Airport|165987|
|Penn Station/Madi...|130699|
+--------------------+------+
only showing top 5 rows


In [44]:
least_frequent = frequent_pickup_zone.tail(1)[0]
print(f'Least frequent pickup zone: {least_frequent.zone} with {least_frequent["count"]} pickups')

Least frequent pickup zone: Arden Heights with 1 pickups


<strong>Arden Heights is the least frequent pickup location zone</strong>